In [ ]:
import pandas as pd

In [ ]:
# Read in DBLP and ACM source tables and the perfect mapping
dblp_df = pd.read_csv('DBLP2.csv', encoding='unicode_escape')
acm_df = pd.read_csv('ACM.csv', encoding='unicode_escape')
mapping_df = pd.read_csv('DBLP-ACM_perfectMapping.csv', encoding='unicode_escape')

# Combine the two source tables into one
combined_df = pd.concat([dblp_df, acm_df], ignore_index=True)

# Build a group_id from the perfect mapping. Each (idDBLP, idACM) pair becomes
# a single group; rows that aren't in the mapping get a singleton group.
mapping_df['group_id'] = mapping_df.index + 1
combined_df = pd.merge(
    combined_df, mapping_df, how='left', left_on='id', right_on='idDBLP'
)
combined_df = pd.merge(
    combined_df, mapping_df, how='left', left_on='id', right_on='idACM',
    suffixes=('_x', '_y')
)
combined_df['group_id'] = combined_df['group_id_x'].fillna(combined_df['group_id_y'])
combined_df = combined_df.drop(
    ['group_id_x', 'group_id_y', 'idDBLP_x', 'idACM_x', 'idDBLP_y', 'idACM_y'],
    axis=1,
)

# Renumber id sequentially, preserving the original id under original_id
combined_df = combined_df.sort_values(by='group_id', ascending=True, na_position='last')
combined_df['original_id'] = combined_df['id']
combined_df['id'] = range(1, len(combined_df) + 1)

combined_df.to_csv('CombinedAcademic.csv', index=False)